# Estimating what AI actually costs

A runnable companion to the ilm.red engineering post
**"The $21 Paragraph: What It Really Costs to Run AI, and Why Your Estimate Is Lying."**

It rebuilds, from scratch and small enough to hold in your hand, the arithmetic behind an AI bill,
and the four-step chain that turned a fourteen-cent job in our own pipeline into a $21.44 charge and
stopped every job on the site for six hours.

1. **The billing unit**: why tokens are not words, and why the same meaning costs different amounts
   in different languages.
2. **The unforecastable half**: output length is decided while the model writes, so the expensive
   part of the bill cannot be known in advance. We simulate it.
3. **Measuring the wrong thing**: reproduce our markup bug on a synthetic post.
4. **The chain**: estimate → no measurement → `COALESCE` → margin, and what it does to a ledger.
5. **The repair**: reprice a ledger from a token log, and find the mis-billed rows with one query.

**Fully offline.** No API keys, no accounts, no network. Standard library plus `matplotlib`.
Licensed Apache-2.0, like the rest of this repository.

In [1]:
# Stdlib only, plus matplotlib for two charts. Nothing here calls a network.
import re, json, random, statistics, base64
from dataclasses import dataclass, field

random.seed(20260806)   # the day of the incident, so the output is deterministic
print("ready, with no API keys required")

ready, with no API keys required


## 1. The billing unit is a fragment, not a word

Real tokenizers learn their vocabulary from data (see Sennrich et al. 2016 for the BPE algorithm).
We do not need a real one to make the point. We need something that behaves the way real ones do:
**common sequences become single tokens, unfamiliar ones get chopped into many.**

The toy below keeps a small "learned" vocabulary of common English fragments. Text it half-knows
falls back to short character runs; text in a script it has no pieces for at all falls back to raw
**UTF-8 bytes**, one token per byte. That second fallback is the one that shows up on invoices:
a non-Latin character is 2-4 bytes, so an unfamiliar script does not merely compress badly, it
multiplies.

In [2]:
COMMON = [
    " the", " and", " to", " of", " a", " in", " is", " it", " you", " that", " for",
    " cost", " token", " model", " price", " bill", " data", "ing", "ion", "ed", "es", "s",
    "The", "This", "We", "AI", " AI",
]
COMMON.sort(key=len, reverse=True)          # longest match first, like a real greedy tokenizer

def tokenize(text, vocab=COMMON, unknown_run=3):
    """Greedy longest-match over a small vocabulary, with a byte-level fallback.

    The fallback is the part that matters. Real tokenizers back off to raw UTF-8 bytes for text
    they have no learned pieces for, and every non-Latin character is 2-4 bytes. So an unfamiliar
    script does not merely fail to compress -- it costs a token per byte.
    """
    out, i = [], 0
    while i < len(text):
        for piece in vocab:
            if text.startswith(piece, i):
                out.append(piece); i += len(piece); break
        else:
            ch = text[i]
            if ord(ch) < 128:
                out.append(text[i:i + unknown_run]); i += unknown_run   # familiar script, poor compression
            else:
                out.extend([ch] * len(ch.encode("utf-8")))              # unfamiliar script, byte fallback
                i += 1
    return out

sample = "The cost of a token is not the cost of a word, and that is the whole problem."
toks = tokenize(sample)
print(f"{len(sample)} characters -> {len(toks)} tokens")
print(f"ratio: {len(sample)/len(toks):.2f} characters per token\n")
print(toks)

77 characters -> 23 tokens
ratio: 3.35 characters per token

['The', ' cost', ' of', ' a', ' token', ' is', ' no', 't t', 'he ', 'cos', 't o', 'f a', ' wo', 'rd,', ' and', ' that', ' is', ' the', ' wh', 'ole', ' pr', 'obl', 'em.']


Now the part that shows up on a multilingual invoice. Below, the *same sentence*: the same meaning,
the same information, in text the toy vocabulary knows well, knows a little, and does not know at
all. Watch the characters-per-token ratio collapse, and with it the price of saying the same thing.

This is a real, measured effect, not an artifact of our toy: Ahia et al. (2023) and Petrov et al.
(2023) document spreads of up to fifteen times across languages on commercial APIs.

In [3]:
VARIANTS = {
    "well represented":   "The cost of a token is not the cost of a word.",
    "partly represented": "Le cout d'un jeton n'est pas le cout d'un mot.",
    "not represented":    "ٹوکن کی لاگت لفظ کی لاگت نہیں ہے۔",
}
RATE_IN = 1.25 / 1_000_000      # a plausible input rate in $/token; structure matters, not the number

print(f"{'variant':22}{'chars':>7}{'tokens':>8}{'chars/tok':>11}{'relative cost':>15}")
base = None
for name, text in VARIANTS.items():
    n = len(tokenize(text))
    base = base or n
    print(f"{name:22}{len(text):>7}{n:>8}{len(text)/n:>11.2f}{n/base:>14.2f}x")

print("\nSame sentence. Same meaning. The meter counts fragments, not ideas.")

variant                 chars  tokens  chars/tok  relative cost
well represented           46      14       3.29          1.00x
partly represented         46      16       2.88          1.14x
not represented            33      31       1.06          2.21x

Same sentence. Same meaning. The meter counts fragments, not ideas.


## 2. The expensive half has not been written yet

Input you can count before you spend anything. Output you cannot: the model decides how long its
answer is *while generating it*, and output is priced several times higher than input.

So a pre-flight estimate is not a measurement with some error bars. It is a **forecast of a random
variable**, and the variable it is forecasting is the expensive one. Predicting response length is
an open research problem (Zheng et al. 2023), which tells you how well your estimator is going to do.

Below we hold the input fixed and let only the output length vary, the way it does in production.

In [4]:
RATE_OUT = 10.00 / 1_000_000    # output costs ~8x input here; asymmetry is the point

def job_cost(n_in, n_out, rate_in=RATE_IN, rate_out=RATE_OUT):
    return n_in * rate_in + n_out * rate_out

N_IN = 4_000                    # a fixed, known prompt
# Output length is right-skewed in practice: usually short, occasionally very long.
samples = [job_cost(N_IN, max(50, int(random.lognormvariate(6.2, 0.85)))) for _ in range(20_000)]

p50, p90, p99 = (statistics.quantiles(samples, n=100)[i] for i in (49, 89, 98))
naive = job_cost(N_IN, 500)     # "assume a typical 500-token answer"

print(f"input is fixed at {N_IN:,} tokens; only the answer varies\n")
print(f"  naive point estimate   ${naive:.5f}")
print(f"  median actual          ${p50:.5f}")
print(f"  90th percentile        ${p90:.5f}   ({p90/naive:.1f}x the estimate)")
print(f"  99th percentile        ${p99:.5f}   ({p99/naive:.1f}x the estimate)")
print(f"  worst seen             ${max(samples):.5f}   ({max(samples)/naive:.1f}x)")
print("\nA single number cannot describe this. Budget a distribution.")

input is fixed at 4,000 tokens; only the answer varies

  naive point estimate   $0.01000
  median actual          $0.00995
  90th percentile        $0.01962   (2.0x the estimate)
  99th percentile        $0.04154   (4.2x the estimate)
  worst seen             $0.15546   (15.5x)

A single number cannot describe this. Budget a distribution.


In [5]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.hist(samples, bins=90, color="#8892a0", edgecolor="none")
for x, lab, col in [(naive, "point estimate", "#ef5033"), (p50, "median", "#2a2a2a"), (p99, "p99", "#2a2a2a")]:
    ax.axvline(x, color=col, linestyle="--" if col == "#2a2a2a" else "-", linewidth=1.6)
    ax.text(x, ax.get_ylim()[1]*0.92, f" {lab}", color=col, fontsize=9, rotation=90, va="top")
ax.set_xlabel("cost of one job (USD)"); ax.set_ylabel("frequency")
ax.set_title("Same prompt, 20,000 times: what one 'typical' estimate hides", fontsize=11)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

## 3. Measuring the wrong thing entirely

Our estimator sized each job by the length of the post **as stored in the database**. That sounds
reasonable right up until you remember what is in a stored post: inline diagrams and base64 image
data, none of which the model ever sees, because the pipeline strips it before calling anything.

Here is a synthetic post with the same shape as the one that broke us, a little prose, a lot of
embedded image.

In [6]:
prose = ("Envelope encryption is a small idea with a big payoff. " * 520).strip()
fake_png = base64.b64encode(bytes(random.getrandbits(8) for _ in range(540_000))).decode()
paths = '<path d="M0 0 L10 10"/>' * 400          # kept out of the f-string: no backslashes allowed in one
stored = (
    f"<p>{prose}</p>\n"
    f'<figure><img src="data:image/png;base64,{fake_png}"/></figure>\n'
    f"<figure><svg viewBox='0 0 100 100'>{paths}</svg></figure>\n"
)

def prose_only(html):
    """What the model actually receives: svg blocks dropped, then all remaining tags."""
    s = re.sub(r"<svg.*?</svg>", "", html, flags=re.S | re.I)
    s = re.sub(r"<[^>]+>", " ", s)
    return re.sub(r"\s+", " ", s).strip()

EST_PER_KCHAR = 0.02
sent = prose_only(stored)
old = len(stored)/1000 * EST_PER_KCHAR
new = len(sent)/1000 * EST_PER_KCHAR

print(f"  stored in the database : {len(stored):>9,} chars")
print(f"  actually sent to model : {len(sent):>9,} chars   ({len(sent)/len(stored):.1%} of stored)")
print()
print(f"  estimate, measuring stored : ${old:>7.2f}")
print(f"  estimate, measuring prose  : ${new:>7.2f}")
print(f"  overstatement              : {old/new:>7.1f}x")

  stored in the database :   757,911 chars
  actually sent to model :    28,599 chars   (3.8% of stored)

  estimate, measuring stored : $  15.16
  estimate, measuring prose  : $   0.57
  overstatement              :    26.5x


## 4. The chain: how a size estimate becomes a charge

A wrong estimate is survivable. What is not survivable is a billing path that lets the estimate
*stand in for a measurement*. Ours had four links:

| link | behaviour |
|---|---|
| 1 | estimate measures stored bytes, not prose |
| 2 | the price lookup is keyed `(task, model)`; the task has no row for that model → per-call cost is `NULL` |
| 3 | nothing priced → the stage reports no true cost → the worker keeps the estimate |
| 4 | `COALESCE(reported, estimated) × margin` → the estimate is now a charge |

Below, each link is a function, so you can switch them on and off and watch the bill move.

In [7]:
@dataclass
class Call:
    model: str; tokens_in: int; tokens_out: int

# Link 2: prices keyed by (task, model). Note there is no ("blog-adapt", ...) row at all.
RATES = {("diacritize", "gemini-2.5-pro"): (1.25, 10.00),
         ("diacritize", "deepseek-chat"):  (0.27,  1.10)}
# What the prices actually are, per model, regardless of task:
BY_MODEL = {"gemini-2.5-pro": (1.25, 10.00), "deepseek-chat": (0.27, 1.10)}

def price_call(task, call, model_fallback: bool):
    rate = RATES.get((task, call.model))
    if rate is None and model_fallback:
        rate = BY_MODEL.get(call.model)          # the fix: price by model, not by task
    if rate is None:
        return None                              # link 2 fires: unpriced
    pin, pout = rate
    return call.tokens_in * pin/1e6 + call.tokens_out * pout/1e6

def run_job(task, calls, estimate, *, model_fallback, margin=0.40):
    priced = [price_call(task, c, model_fallback) for c in calls]
    measured = [p for p in priced if p is not None]
    # Link 3: a stage with nothing priced reports nothing.
    reported = sum(measured) if measured else None
    # Link 4: COALESCE(reported, estimate) -- the guess becomes the bill.
    true_cost = reported if reported is not None else estimate
    return {"reported": reported, "true_cost": true_cost,
            "billed": round(true_cost * (1 + margin), 6),
            "unpriced_calls": priced.count(None)}

calls = [Call("gemini-2.5-pro", 17_552, 10_563), Call("deepseek-chat", 11_749, 6_942)]

broken = run_job("blog-adapt", calls, estimate=15.3123, model_fallback=False)
fixed  = run_job("blog-adapt", calls, estimate=0.5500,  model_fallback=True)

print("BEFORE  (task-keyed prices, estimate falls through to the bill)")
print(f"   reported cost : {broken['reported']}")
print(f"   billed        : ${broken['billed']:.4f}   <-- the $21.44\n")
print("AFTER   (model-level fallback, real measurement reported)")
print(f"   reported cost : ${fixed['reported']:.4f}")
print(f"   billed        : ${fixed['billed']:.4f}")
print(f"\n   overstatement removed: {broken['billed']/fixed['billed']:.0f}x")

BEFORE  (task-keyed prices, estimate falls through to the bill)
   reported cost : None
   billed        : $21.4372   <-- the $21.44

AFTER   (model-level fallback, real measurement reported)
   reported cost : $0.1384
   billed        : $0.1937

   overstatement removed: 111x


Two details in that code worth pausing on, because both are the kind of thing that survives review:

- **The fallback is silent, and it points the wrong way.** When measurement fails, the code quietly
  substitutes the *less* accurate number. A loud failure here would have cost one afternoon; a quiet
  one cost six hours of queue.
- **The margin multiplies the error.** Applying a markup to a measured cost is fine. Applying it to a
  guess turns a 15x modelling error into a 21x charge.

## 5. Why a cost bug is an availability bug

Spend caps get enforced where the money is spent, which is usually the hot path. Ours is checked in
the first statement of the function that hands work to workers, so once the day's *recorded* spend
crossed the cap, that function paused itself and returned empty to every worker, for every kind of
job on the site.

In [8]:
DAILY_CAP = 50.00

def simulate_day(bill_per_job, n_jobs=328, cap=DAILY_CAP):
    spent = 0.0
    for i in range(1, n_jobs + 1):
        if spent >= cap:
            return i - 1, spent, True          # paused: every later job, of every kind, blocked
        spent += bill_per_job
    return n_jobs, spent, False

# Use a representative per-job figure, not the single most expensive stage: across the real
# incident the measured bill averaged about 10 cents a job, against estimates 50x higher.
AVG_MEASURED, AVG_ESTIMATED = 0.102, 5.20
for label, per_job in [("billed from the estimate", AVG_ESTIMATED),
                       ("billed from measurement",  AVG_MEASURED)]:
    done, spent, paused = simulate_day(per_job)
    state = f"PAUSED after {done} of 328" if paused else f"completed all {done}"
    print(f"  {label:26} ${per_job:>7.4f}/job  ->  {state}  (${spent:.2f} of ${DAILY_CAP:.0f})")

print("\nSame work. Same real spend. One of them takes the whole site down.")

  billed from the estimate   $ 5.2000/job  ->  PAUSED after 10 of 328  ($52.00 of $50)
  billed from measurement    $ 0.1020/job  ->  completed all 328  ($33.46 of $50)

Same work. Same real spend. One of them takes the whole site down.


## 6. The repair: find the lie, then price it from the log

The diagnostic that actually found this was one column. A stage that reports its own measured cost
populates a stats field; a stage that does not, leaves it empty, and bills its estimate. That
difference is queryable, which is the whole point.

Below is the ledger as we found it, and the same repair we ran in production: reprice every affected
row from the token log, and compute what has to be refunded.

In [9]:
ledger = [   # kind, reported_stats?, estimate, billed, calls
  ("blog-adapt",     False, 15.3123, 21.4372, [Call("gemini-2.5-pro", 17_552, 10_563), Call("deepseek-chat", 11_749, 6_942)]),
  ("blog-translate", False, 15.3123, 21.4372, [Call("gemini-2.5-pro",  9_120,  4_880), Call("deepseek-chat",  8_400, 3_100)]),
  ("blog-adapt",     False,  3.4704,  4.8586, [Call("gemini-2.5-pro", 12_400,  7_900)]),
  ("diacritize",     True,   0.0400,  0.0776, [Call("deepseek-chat",   9_800,  4_100)]),
]

print(f"{'kind':16}{'stats':>7}{'billed':>10}{'measured':>11}{'should be':>11}{'delta':>10}")
refund = 0.0
for kind, has_stats, est, billed, calls in ledger:
    measured = sum(price_call(kind, c, model_fallback=True) for c in calls)
    should = round(measured * 1.40, 6)
    refund += billed - should
    flag = "yes" if has_stats else "NULL"
    print(f"{kind:16}{flag:>7}{billed:>10.4f}{measured:>11.4f}{should:>11.4f}{billed-should:>10.4f}")

print(f"\n  total over-billed: ${refund:.4f}")
print("  rows with stats = NULL are the ones that billed a guess. That is the whole query.")

kind              stats    billed   measured  should be     delta
blog-adapt         NULL   21.4372     0.1384     0.1937   21.2435
blog-translate     NULL   21.4372     0.0659     0.0922   21.3450
blog-adapt         NULL    4.8586     0.0945     0.1323    4.7263
diacritize          yes    0.0776     0.0072     0.0100    0.0676

  total over-billed: $47.3823
  rows with stats = NULL are the ones that billed a guess. That is the whole query.


## What to take away

1. **Never let an estimate and a measurement share a column.** If your code has a fallback from
   measured to estimated cost, you cannot answer "which of my charges are real?" after the fact.
2. **Report nothing rather than zero.** A stage that could not price its calls must say so. "Free"
   and "unknown" are different facts and must not collapse into the same number.
3. **Make "this was measured" queryable.** One boolean per row turns a forensic exercise into a
   `WHERE` clause.
4. **Scope your caps.** A global cap in the claim path converts any single stage's accounting bug
   into a site-wide outage.
5. **Budget a distribution.** The expensive half of an AI bill is a random variable that does not
   exist until you have paid for it.

Teaching code, not production code: the goal is to make the moving parts obvious. The real system
is the same arithmetic wired into a job queue with a token log per call.